# 4. Outlier Detection and Treatment

This notebook covers:
1. **Statistical Outlier Detection:**
   - **IQR (Interquartile Range) Method** (Best for skewed/non-normal data).
   - **Z-Score Method** (Best for approximately normal/bell-shaped data).
2. **Algorithmic Outlier Detection:**
   - **Isolation Forest** (Multi-dimensional outlier detection).
3. **Outlier Treatment Strategies:**
   - Capping / Truncation (Winsorization) vs. Dropping.
4. **Data Leakage Rule:** Computing outlier thresholds strictly on the training set.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a sample dataset containing realistic values + clear outliers
np.random.seed(42)
normal_salaries = np.random.normal(loc=60000, scale=15000, size=100)
outlier_salaries = [250000, 300000, 5000] # Extreme high and low points

salary_data = np.concatenate([normal_salaries, outlier_salaries])
df_outliers = pd.DataFrame({'Salary': salary_data})

print("Dataset size:", len(df_outliers))
display(df_outliers.describe())

Dataset size: 103


,Salary
count,103.000000
mean,62128.448776
std,33558.957676
min,5000.000000
25%,50982.410234
50%,58265.275764
75%,67574.861896
max,300000.000000


---
## Part 1: Statistical Method — IQR (Interquartile Range)

- **$Q_1$ (25th percentile)** and **$Q_3$ (75th percentile)**
- $\text{IQR} = Q_3 - Q_1$
- $\text{Lower Bound} = Q_1 - (1.5 \times \text{IQR})$
- $\text{Upper Bound} = Q_3 + (1.5 \times \text{IQR})$

Any value outside $[\text{Lower Bound}, \text{Upper Bound}]$ is flagged as an outlier.

In [2]:
Q1 = df_outliers['Salary'].quantile(0.25)
Q3 = df_outliers['Salary'].quantile(0.75)

IQR = Q3 - Q1

lower_bound_iqr = Q1 - 1.5 * IQR 
upper_bound_iqr = Q3 + 1.5 * IQR

print(f"IQR Lower limit {lower_bound_iqr:.2f}")
print(f"IQR Upper limit {upper_bound_iqr:.2f}")

iqr_outliers = df_outliers[(df_outliers['Salary'] < lower_bound_iqr) | (df_outliers['Salary'] > upper_bound_iqr)]

print(f'outliers detected: {len(iqr_outliers)}')
display(iqr_outliers)

IQR Lower limit 26093.73
IQR Upper limit 92463.54
outliers detected: 4


,Salary
74,20703.823439
100,250000.000000
101,300000.000000
102,5000.000000


---
## Part 2: Statistical Method — Z-Score

- Measures how many standard deviations ($\sigma$) a data point is from the mean ($\mu$).
- Standard cutoff: $|Z| > 3$ is typically flagged as an outlier.
- **Limitation:** Highly sensitive to extreme values because mean and standard deviation themselves get distorted by outliers.

In [3]:
mean = df_outliers['Salary'].mean()
std = df_outliers['Salary'].std()

df_outliers['z_score'] = (df_outliers['Salary'] - mean) / std

z_outliers = df_outliers[df_outliers['z_score'].abs() > 3]

print(f'detected {len(z_outliers)} outliers using the z score method')
display(z_outliers)

detected 2 outliers using the z score method


,Salary,z_score
100,250000.0,5.598253
101,300000.0,7.088169


---
## Part 3: Algorithmic Method — Isolation Forest

- **Isolation Forest** works by isolating anomalies using random decision tree partitions.
- Outliers require fewer splits to isolate compared to normal points.
- **Advantage:** Works across multiple numerical features simultaneously (multivariate).

In [4]:
from sklearn.ensemble import IsolationForest

# Initialize IsolationForest (contamination is expected % of outliers)
iso_forest = IsolationForest(contamination=0.03, random_state=42)
df_outliers['Anomaly_Score'] = iso_forest.fit_predict(df_outliers[['Salary']])

# -1 indicates an anomaly/outlier, 1 indicates normal
iso_outliers = df_outliers[df_outliers['Anomaly_Score'] == -1]
print(f"Detected {len(iso_outliers)} anomalies using Isolation Forest:")
display(iso_outliers)

Detected 4 anomalies using Isolation Forest:


,Salary,z_score,Anomaly_Score
74,20703.823439,-1.234384,-1
100,250000.000000,5.598253,-1
101,300000.000000,7.088169,-1
102,5000.000000,-1.702331,-1


---
## Part 4: Treatment — Capping (Winsorization) vs. Dropping

- **Dropping:** Deletes the row. Use **only** if the outlier is an impossible error (e.g., negative age).
- **Capping (Winsorization):** Clamps values outside the threshold to the lower and upper limits. This retains the row and dataset size while neutralising extreme leverage.

#### 1. Capping

If you see the output, we can see the capped max and min salary, what does that mean now:

If any person's salary is **greater than** `300000.0`, it will be replaced with the value `92463.53938900332` no matter how many person there are with salary greater than that value, as its the **upper_limit_iqr** (named as upper bound iqr) which we calculated above according to **Interquartile Range** process, and if salary **lesser than** `5000.0` is replaced with the `26093.732740819585` similarly to the greater than condition. 

> Capping is all about replacing values, its important here. You'll understand this in the next explanation cell related to `Dropping` process

In [6]:
# Apply Capping (Winsorization) using IQR bounds
df_capped = df_outliers.copy()
df_capped['Salary_Capped'] = df_capped['Salary'].clip(lower=lower_bound_iqr, upper=upper_bound_iqr)

# Verify capping effect
print("Original Max Salary:", df_outliers['Salary'].max())
print("Capped Max Salary:", df_capped['Salary_Capped'].max())

print("\nOriginal Min Salary:", df_outliers['Salary'].min())
print("Capped Min Salary:", df_capped['Salary_Capped'].min())

Original Max Salary: 300000.0
Capped Max Salary: 92463.53938900332

Original Min Salary: 5000.0
Capped Min Salary: 26093.732740819585


#### 2. Dropping

As the word, as the work, we just drop the row simply instead of replacing the salary with some calculated value. If you think logically, you may get a point like, **deleting that row with salary which is a outlier is better than morphing the salary**, as its `artificial salary` right, not the true value, its calculated!. But it's quite opposite in real world.


In [ ]:
# Keep ONLY the rows where Salary is within the normal boundaries, while deleting the outliers
df_dropped = df_outliers[(df_outliers['Salary'] >= lower_bound_iqr) & (df_outliers['Salary'] <= upper_bound_iqr)].copy()

In real world, morphing the data is considered better than totally removing the row, here's why: 

Why Capping is Used Most of the Time in ML
1. Preventing "Collateral Data Loss"
Real datasets have many features (e.g., 50 columns). If 5% of your rows have an outlier in Salary, another 5% in Age, and another 5% in Transaction Count, dropping rows can easily delete 20% to 30% of your entire training dataset. Capping retains 100% of your training volume.

2. Preserving the Monotonic Signal
A machine learning model (like Linear Regression or Gradient Boosting) doesn't need to know the exact multi-million-dollar number to know that someone is in the highest bracket. Capping a ₹12,00,000 salary to ₹1,50,000 still correctly positions that user as a high-earner relative to everyone else, preserving the core predictive pattern without destabilizing gradient descent or distance metrics.

3. Real-Time Production Feasibility
When your trained model is deployed as an API, a live user might submit an extreme value. You cannot run df.dropna() on a live paying user. You must process their request by clipping the value to your model's maximum trained boundary (df['Salary'].clip(lower_limit, upper_limit)) and returning an instant prediction.

Why Not Always Use Dropping?

Dropping should be reserved strictly for noise, corrupted data, and data collection failures:

- Negative numbers where only positive are allowed (Age = -5, Price = -100).
- Impossible biological or physical bounds (Body_Temp = 200°C).
- Bot/Spam traffic that should not be in the training distribution at all.

For genuine real-world extremes, capping (winsorization) is the default standard in production machine learning pipelines.